# Audit minute-bar gaps

## Goal

Produce interval boundaries, observed closes and missingness flags. This audits time coverage; it neither estimates volatility nor fabricates bars.

This notebook uses synthetic teaching data, not a paper replication or production observations.

## Setup

Use a Python 3.10+ kernel and run all cells in order. Computation uses only the standard library, without keys, networking or extra data files. Open in an existing Jupyter environment.

Embedded inputs match inputs.json in the same download directory. Edit args in the next cell to experiment; preserve explicit times and units.

In [ ]:
import json

# Synthetic inputs; no credentials or network access.
bundle = json.loads("{\"version\":1,\"tutorial\":\"minute-bar-gaps\",\"identity\":\"synthetic\",\"args\":[[{\"openTime\":\"2025-01-06T01:30:00Z\",\"close\":100},{\"openTime\":\"2025-01-06T01:40:00Z\",\"close\":102}],[\"2025-01-06T01:30:00Z\",\"2025-01-06T01:35:00Z\",\"2025-01-06T01:40:00Z\"],5],\"expected\":[{\"openTime\":\"2025-01-06T01:30:00.000Z\",\"endExclusive\":\"2025-01-06T01:35:00.000Z\",\"close\":100,\"status\":\"observed\"},{\"openTime\":\"2025-01-06T01:35:00.000Z\",\"endExclusive\":\"2025-01-06T01:40:00.000Z\",\"close\":null,\"status\":\"missing\"},{\"openTime\":\"2025-01-06T01:40:00.000Z\",\"endExclusive\":\"2025-01-06T01:45:00.000Z\",\"close\":102,\"status\":\"observed\"}]}")
args = bundle["args"]
expected = bundle["expected"]
print(json.dumps(args, ensure_ascii=False, indent=2))

## Steps

### 1. Freeze timestamp semantics

Check whether vendor timestamps label interval starts or ends, then normalize explicitly to UTC whole seconds. The example uses half-open intervals; endExclusive does not replace the vendor's raw close_time. Preserve original timezone and boundary semantics rather than merely renaming a column.

### 2. Prepare an independent grid

Supply expectedOpens from the relevant calendar and session rules, including only finished intervals expected to be observed. Lunch breaks, holidays and closures are not gaps; suspensions need additional security-level status. Deriving the grid from available bars would hide missing intervals before the audit.

### 3. Join exactly and reject conflicts

Join on exact starts; reject duplicate bars, overlapping grid intervals, off-grid observations and invalid prices. Gaps between sessions are allowed, overlapping intervals are not. The function does not guess a nearest minute or combine securities, so partition by instrument and venue first.

### 4. Preserve gaps before explaining them

The synthetic grid contains three five-minute intervals with the middle observation absent; output is missing with a null close. That does not identify suspension, no trading or acquisition failure. Check receipts and market state separately, and disclose any treatment before computing returns across gaps.

### Method and assumptions

- Do not audit unfinished candles; no trade is not automatically an ingestion failure.
- An exchange calendar does not establish a security's suspension status.
- The example performs no interpolation, zero filling or forward filling.

In [ ]:
from datetime import datetime, timedelta, timezone
import math


def audit_bar_grid(rows, expected_opens, interval_minutes):
    def parse(value):
        try:
            parsed = datetime.strptime(value, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
            if parsed.strftime("%Y-%m-%dT%H:%M:%SZ") != value:
                raise ValueError()
            return parsed
        except (ValueError, TypeError):
            raise ValueError("utc_seconds_required")

    if isinstance(interval_minutes, bool) or not isinstance(interval_minutes, (int, float)) or not math.isfinite(interval_minutes) or int(interval_minutes) != interval_minutes or not 0 < interval_minutes <= 1440:
        raise ValueError("invalid_interval")
    if not expected_opens or len(expected_opens) > 10000:
        raise ValueError("invalid_grid")
    grid = sorted(map(parse, expected_opens))
    interval = timedelta(minutes=interval_minutes)
    if any(current - previous < interval for previous, current in zip(grid, grid[1:])):
        raise ValueError("overlapping_grid")
    slots, observations = set(grid), {}
    for row in rows:
        time = parse(row.get("openTime"))
        if time not in slots:
            raise ValueError("outside_grid")
        if time in observations:
            raise ValueError("duplicate_bar")
        close = row.get("close")
        if isinstance(close, bool) or not isinstance(close, (int, float)) or not math.isfinite(close) or close <= 0:
            raise ValueError("invalid_close")
        observations[time] = close
    iso = lambda value: value.isoformat(timespec="milliseconds").replace("+00:00", "Z")
    return [{"openTime": iso(time), "endExclusive": iso(time + interval), "close": observations.get(time), "status": "observed" if time in observations else "missing"} for time in grid]


### Run the sample

Three rows: observed, missing, observed; closes are 100, null, 102. The two source rows remain unchanged.

In [ ]:
result = audit_bar_grid(*args)
print(json.dumps(result, ensure_ascii=False, indent=2))

## Checks

Compare every row with the browser example's expected output. After editing inputs, a failed assertion may be expected: explain the difference before changing the check.

In [ ]:
assert result == expected, "Output differs from the reference synthetic example"
assert bundle["identity"] == "synthetic"
print("Passed: output matches the synthetic browser example.")

## Next steps

Before real data, confirm grants, fields, schema_major, windows and provenance using authenticated GET /v1/catalog, then map the actual contract. Candidate IDs below do not guarantee availability or historical completeness. API as_of is not a historical filing-version guarantee. Validate again after substituting real inputs; the synthetic pass does not transfer.

- `cn.dataset.stk_mins`
- `cn.market.trade_calendar`

### References

- [Tushare: historical stock minutes](https://tushare.pro/document/2?doc_id=370)
- [Andersen et al.: realized-volatility construction](https://users.ssc.wisc.edu/~behansen/718/Anderson2003.pdf)

[Back to tutorial](https://tradingdatas.com/recipes/minute-bar-gaps/)